# 4.25 — Kernel Density Estimation
Kernel Density Estimation turns unlabeled data into structure by choosing the right notion of similarity, compression, or surprise. Probability normalization and distance-based kernels become the way local influence is measured. Save a copy to Drive to edit.

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import SpectralBiclustering
from sklearn.datasets import load_digits, load_iris, make_blobs, make_moons
from sklearn.ensemble import IsolationForest
from sklearn.metrics import adjusted_rand_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KernelDensity, LocalOutlierFactor
from sklearn.preprocessing import MinMaxScaler, StandardScaler


def cluster_ladder():
    """D1..D5 clustering ladder of rising difficulty. Returns [(name, X, y_true, k), ...]."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.3, 0.2], [3.0, 3.0], [3.2, 2.8], [0.1, 3.1], [0.2, 2.9]])
    y1 = np.array([0, 0, 1, 1, 2, 2])
    rungs.append(("D1 hand 3 clusters", x1, y1, 3))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.7, random_state=1)
    rungs.append(("D2 clean blobs", x2, y2, 3))

    x3, y3 = make_blobs(n_samples=240, centers=3, cluster_std=1.6, random_state=2)
    transform = np.array([[0.6, -0.6], [-0.4, 0.8]])
    x3 = x3 @ transform
    rungs.append(("D3 anisotropic + overlap", x3, y3, 3))

    iris = load_iris()
    rungs.append(("D4 Iris (real, 4-D)", iris.data, iris.target, 3))

    digits = load_digits()
    keep = np.isin(digits.target, [0, 1, 2, 3])
    rungs.append(("D5 digits 0-3 (real, 64-D)", digits.data[keep] / 16.0, digits.target[keep], 4))

    return rungs


def matrix_for_biclustering(X):
    X = np.asarray(X, dtype=float)
    return MinMaxScaler().fit_transform(X) + 0.001


def project_for_plot(X):
    X = np.asarray(X, dtype=float)
    if X.shape[1] == 2:
        return X
    centered = StandardScaler().fit_transform(X)
    u, s, vt = np.linalg.svd(centered, full_matrices=False)
    return centered @ vt[:2].T


def preview_rungs(rungs):
    for name, X, y, k in rungs:
        counts = dict(zip(*np.unique(y, return_counts=True)))
        print(name, "shape=", X.shape, "k=", k, "labels=", counts)
        print(np.round(X[:3, : min(5, X.shape[1])], 3))


def make_anomaly_ladder():
    rungs = []

    X1 = np.array([[0.0, 0.0], [0.2, 0.0], [0.0, 0.1], [4.0, 4.0]])
    y1 = np.array([0, 0, 0, 1])
    rungs.append(("D1 hand normals + one outlier", X1, y1, 2))

    for idx, (name, X, labels, k) in enumerate(cluster_ladder()[1:], start=2):
        rng = np.random.default_rng(240 + idx)
        X = np.asarray(X, dtype=float)
        n_outliers = max(8, int(0.08 * len(X)))
        lo = X.min(axis=0)
        hi = X.max(axis=0)
        span = np.maximum(hi - lo, 1.0)
        low_outliers = lo - rng.uniform(1.5, 2.5, size=(n_outliers // 2, X.shape[1])) * span
        high_outliers = hi + rng.uniform(1.5, 2.5, size=(n_outliers - n_outliers // 2, X.shape[1])) * span
        outliers = np.vstack([low_outliers, high_outliers])
        X_aug = np.vstack([X, outliers])
        y_aug = np.concatenate([np.zeros(len(X), dtype=int), np.ones(n_outliers, dtype=int)])
        rungs.append((name.replace("clusters", "normal groups") + " + synthetic outliers", X_aug, y_aug, 2))

    return rungs


def make_transaction_ladder():
    ladders = []
    base = [
        {"bread", "milk"},
        {"bread", "milk", "eggs"},
        {"bread", "eggs"},
        {"milk", "eggs"},
    ]
    labels = np.array([0, 0, 1, 1])
    ladders.append(("D1 four baskets", base, labels, 2))

    for name, X, y, k in cluster_ladder()[1:]:
        X = StandardScaler().fit_transform(X)
        transactions = []
        for row in X:
            items = set()
            for j, value in enumerate(row[: min(8, X.shape[1])]):
                if value <= -0.5:
                    items.add(f"f{j}_low")
                elif value >= 0.5:
                    items.add(f"f{j}_high")
                else:
                    items.add(f"f{j}_mid")
            transactions.append(items)
        ladders.append((name + " discretized transactions", transactions, y, k))

    return ladders


def print_table(rows, metric_name):
    print(f"{'rung':<38} {metric_name:>12}")
    for name, value in rows:
        print(f"{name:<38} {value:>12.3f}")


## The concept, built once (D1): kernel weights
The lesson's normalized influence is $$p_i=\frac{\exp(-d_i^2/2h^2)}{\sum_j\exp(-d_j^2/2h^2)}.$$ We compute the exponentials before wrapping the same idea in `KernelDensity`.

In [ ]:

def gaussian_kernel_weights(distances, bandwidth):
    distances = np.asarray(distances, dtype=float)
    weights = np.exp(-(distances ** 2) / (2.0 * bandwidth ** 2))
    probabilities = weights / weights.sum()
    return weights, probabilities

weights, probabilities = gaussian_kernel_weights([1.0, 0.0, 1.0], bandwidth=1.0)
print("lesson exponentials", np.round(weights, 3))
print("lesson normalized weights", np.round(probabilities, 3))
assert np.allclose(np.round(weights, 3), np.array([0.607, 1.000, 0.607]))


## The concept, built once (D1): fit and score KDE
The reusable method fits a Gaussian KDE and returns log-likelihood scores. Higher held-out average log-likelihood means the density model assigned more probability to unseen data.

In [ ]:

def method(X, bandwidth):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kde = KernelDensity(kernel="gaussian", bandwidth=bandwidth)
    kde.fit(X_scaled)
    log_density = kde.score_samples(X_scaled)
    return kde, scaler, log_density

weight_sum = float(weights.sum())
middle_influence = float(probabilities[1])
print("lesson weight sum", round(weight_sum, 3))
print("lesson middle influence", round(middle_influence, 3))
assert round(weight_sum, 3) == 2.213
assert round(middle_influence, 3) == 0.452


## The dataset ladder
The same F2 ladder is scored with KDE. Hidden labels are not used by KDE; train/test split gives the log-likelihood metric.

In [ ]:

rungs = cluster_ladder()
preview_rungs(rungs)


## Run the same method across D1–D5
We fit KDE on a training split and report average held-out log-likelihood across rungs.

In [ ]:

rows = []
artifacts = []
for name, X, y_true, k in rungs:
    X_train, X_test = train_test_split(X, test_size=0.35, random_state=0)
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(X_train)
    test_scaled = scaler.transform(X_test)
    bandwidth = 0.6 if X.shape[1] <= 4 else 1.2
    kde = KernelDensity(kernel="gaussian", bandwidth=bandwidth)
    kde.fit(train_scaled)
    train_log = kde.score_samples(train_scaled)
    test_log = kde.score_samples(test_scaled)
    metric = float(test_log.mean())
    rows.append((name, metric))
    artifacts.append((name, X, y_true, bandwidth, train_log, test_log, kde, scaler))
print_table(rows, "avg logL")


## Results visualization
For 2-D views we draw density-colored projections; for all rungs we track held-out log-likelihood.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
flat_axes = axes.ravel()
for ax, artifact in zip(flat_axes, artifacts):
    name, X, y_true, bandwidth, train_log, test_log, kde, scaler = artifact
    Z = project_for_plot(X)
    if X.shape[1] == 2:
        X_scaled = scaler.transform(X)
        scores = kde.score_samples(X_scaled)
    else:
        scores = np.interp(np.arange(len(Z)), np.linspace(0, len(Z) - 1, len(train_log)), np.sort(train_log))
    ax.scatter(Z[:, 0], Z[:, 1], c=scores[: len(Z)], cmap="viridis", s=18)
    ax.set_title(name.split("(")[0] + f"\nh={bandwidth}")
    ax.set_xlabel("view 1")
    ax.set_ylabel("view 2")
flat_axes[-1].plot(range(1, 6), [value for _, value in rows], marker="o")
flat_axes[-1].set_xticks(range(1, 6))
flat_axes[-1].set_xlabel("rung")
flat_axes[-1].set_ylabel("held-out avg logL")
flat_axes[-1].set_title("KDE log-likelihood vs. complexity")
plt.tight_layout()


## Pitfall on D5: bandwidth tells the story
Too-small bandwidth memorizes spikes; too-large bandwidth flattens the digit density. A validation sweep chooses a middle value instead of trusting either training score extreme.

In [ ]:

name, X_d5, y_d5, _ = rungs[-1]
X_train, X_test = train_test_split(X_d5, test_size=0.35, random_state=0)
scaler = StandardScaler()
train_scaled = scaler.fit_transform(X_train)
test_scaled = scaler.transform(X_test)
bandwidths = [0.05, 0.2, 0.6, 1.2, 2.4]
validation = []
for bandwidth in bandwidths:
    kde = KernelDensity(kernel="gaussian", bandwidth=bandwidth)
    kde.fit(train_scaled)
    validation.append((bandwidth, float(kde.score_samples(test_scaled).mean())))
best_bandwidth, best_score = max(validation, key=lambda item: item[1])
print("validation sweep", [(b, round(s, 3)) for b, s in validation])
print("fixed bandwidth", best_bandwidth, "held-out logL", round(best_score, 3))


## Evaluate it + Practice
- Compare the rung metric with a no-skill baseline: random row labels, random anomaly scores, one global density, or independence-only rules.
- Run a cheap sanity check: shuffle one key input and confirm the metric or discovered artifact degrades.
- Ablate the key idea: remove scaling, held-out scoring, bandwidth selection, or lift filtering and watch the metric drop.
- Inspect failure signals: unstable assignments, high training-only fit, score thresholds that flag whole subgroups, or rules with high support but lift near 1.
- Keep the hidden labels for evaluation only; the unsupervised method must not see them during fitting.

Practice prompts:
1. Change one hyperparameter near the default and predict which rung is most sensitive.


In [ ]:
# Your experiment for practice prompt 1 goes here.

2. Add a scaling or stability check and describe the before/after metric.

In [ ]:
# Your experiment for practice prompt 2 goes here.

3. Replace the D1 toy data with your own four-to-six point example and keep the asserts auditable.

In [ ]:
# Your experiment for practice prompt 3 goes here.